# Aula 2 — Treinando Redes Neurais Profundas (Keras / TensorFlow)

Notebook de acompanhamento da Aula 2 — **Tópicos Avançados em IA · UFABC**  
Adaptado de MIT 15.773 (Farias & Ramakrishnan, 2024)

---

## Índice

1. [Configuração](#1-configuração)
2. [Parte 1 — Dataset: Doença Cardíaca](#2-parte-1--dataset-doença-cardíaca)
3. [Parte 2 — Definindo o Modelo](#3-parte-2--definindo-o-modelo)
4. [Parte 3 — Loss e Otimizador](#4-parte-3--loss-e-otimizador)
5. [Parte 4 — Treinamento com Early Stopping e Dropout](#5-parte-4--treinamento-com-early-stopping-e-dropout)
6. [Parte 5 — Avaliação](#6-parte-5--avaliação)
7. [Parte 6 — Fashion-MNIST (Multiclasse)](#7-parte-6--fashion-mnist-multiclasse)

## 1. Configuração

In [ ]:
# Instala dependências (necessário no Colab)
!pip install -q ucimlrepo tensorflow

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

## 2. Parte 1 — Dataset: Doença Cardíaca

Usamos o **Cleveland Clinic Heart Disease Dataset** (UCI ML Repository, id=45):  
~300 pacientes, 13 atributos clínicos → saída binária (doença: sim/não).

Após *one-hot encoding* das variáveis categóricas, chegamos a **29 entradas**.

In [ ]:
from ucimlrepo import fetch_ucirepo

heart = fetch_ucirepo(id=45)
X_raw = heart.data.features
y_raw = heart.data.targets

print(f"Exemplos: {X_raw.shape[0]}  |  Features brutas: {X_raw.shape[1]}")
X_raw.head()

In [ ]:
# Alvo: 0 = sem doença, 1 = com doença (qualquer grau > 0)
y = (y_raw.values.ravel() > 0).astype(np.float32)

# Preenche valores ausentes com a mediana
X_raw = X_raw.fillna(X_raw.median(numeric_only=True))

# One-hot encoding das colunas categóricas
cat_cols = ['cp', 'restecg', 'slope', 'thal']          # variáveis nominais
X = pd.get_dummies(X_raw, columns=cat_cols, drop_first=False)

print(f"Features após one-hot: {X.shape[1]}")
print(f"Proporção de casos positivos: {y.mean():.1%}")

In [ ]:
# Split treino / validação / teste  (70 / 15 / 15)
X_train, X_temp, y_train, y_temp = train_test_split(
    X.values.astype(np.float32), y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# Normalização: ajusta apenas no treino
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

n_features = X_train.shape[1]
print(f"Treino: {X_train.shape}  |  Val: {X_val.shape}  |  Teste: {X_test.shape}")
print(f"Número de features (entradas da rede): {n_features}")

## 3. Parte 2 — Definindo o Modelo

Arquitetura: **entrada → 16 ReLU → dropout → 1 sigmoide**

$$
\underbrace{n \times 16}_{W^{(1)}} + \underbrace{16}_{b^{(1)}} + \underbrace{16 \times 1}_{W^{(2)}} + \underbrace{1}_{b^{(2)}}
$$

In [ ]:
inp = keras.Input(shape=(n_features,))
h   = keras.layers.Dense(16, activation='relu')(inp)
h   = keras.layers.Dropout(0.3)(h)
out = keras.layers.Dense(1, activation='sigmoid')(h)
model = keras.Model(inp, out)

model.summary()

## 4. Parte 3 — Loss e Otimizador

| Tipo de saída | Loss recomendada |
|---|---|
| Binária (sigmoide) | `binary_crossentropy` |
| Multiclasse (softmax) | `categorical_crossentropy` / `sparse_categorical_crossentropy` |
| Regressão | `mean_squared_error` |

In [ ]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy'],
)

## 5. Parte 4 — Treinamento com Early Stopping e Dropout

**Early Stopping**: monitora a loss de validação e interrompe o treino quando ela para de cair, restaurando os pesos do melhor momento.

**Dropout**: a cada passo de treino desliga aleatoriamente uma fração dos neurônios, forçando a rede a distribuir o aprendizado. Desativado na inferência.

In [ ]:
es = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1,
)

hist = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=32,
    callbacks=[es],
    verbose=0,
)

print(f"Parou na época {len(hist.history['loss'])}")

In [ ]:
# Curvas de loss
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(hist.history['loss'],     label='Treino')
axes[0].plot(hist.history['val_loss'], label='Validação')
axes[0].set_title('Loss')
axes[0].set_xlabel('Época')
axes[0].legend()

axes[1].plot(hist.history['accuracy'],     label='Treino')
axes[1].plot(hist.history['val_accuracy'], label='Validação')
axes[1].set_title('Acurácia')
axes[1].set_xlabel('Época')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Parte 5 — Avaliação

In [ ]:
loss_test, acc_test = model.evaluate(X_test, y_test, verbose=0)
print(f"Loss no teste : {loss_test:.4f}")
print(f"Acurácia no teste: {acc_test:.1%}")

In [ ]:
y_pred = (model.predict(X_test, verbose=0) >= 0.5).astype(int).ravel()

print(classification_report(y_test, y_pred, target_names=['Sem doença', 'Com doença']))

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=['Sem doença', 'Com doença']
).plot(cmap='Blues')
plt.title('Matriz de Confusão — Conjunto de Teste')
plt.show()

## 7. Parte 6 — Fashion-MNIST (Multiclasse)

Classificação de 10 categorias de roupas a partir de imagens 28×28.  
Arquitetura: **Flatten → 128 ReLU → Dropout → 64 ReLU → 10 Softmax**

> `sparse_categorical_crossentropy` aceita rótulos como inteiros (sem one-hot).

In [ ]:
(X_ftr, y_ftr), (X_fte, y_fte) = keras.datasets.fashion_mnist.load_data()

# Normaliza para [0, 1]
X_ftr = X_ftr / 255.0
X_fte = X_fte / 255.0

class_names = ['T-shirt','Calça','Pulôver','Vestido','Casaco',
               'Sandália','Camisa','Tênis','Bolsa','Bota']

print(f"Treino: {X_ftr.shape}  |  Teste: {X_fte.shape}")

# Visualiza alguns exemplos
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(X_ftr[i], cmap='gray')
    ax.set_title(class_names[y_ftr[i]])
    ax.axis('off')
plt.suptitle('Amostras do Fashion-MNIST')
plt.tight_layout()
plt.show()

In [ ]:
inp_f  = keras.Input(shape=(28, 28))
x      = keras.layers.Flatten()(inp_f)
x      = keras.layers.Dense(128, activation='relu')(x)
x      = keras.layers.Dropout(0.3)(x)
x      = keras.layers.Dense(64, activation='relu')(x)
out_f  = keras.layers.Dense(10, activation='softmax')(x)

model_f = keras.Model(inp_f, out_f)
model_f.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy'],
)
model_f.summary()

In [ ]:
es_f = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, verbose=1)

hist_f = model_f.fit(
    X_ftr, y_ftr,
    validation_split=0.1,
    epochs=30,
    batch_size=64,
    callbacks=[es_f],
    verbose=0,
)

_, acc_f = model_f.evaluate(X_fte, y_fte, verbose=0)
print(f"Acurácia no teste (Fashion-MNIST): {acc_f:.1%}")

In [ ]:
# Curva de acurácia
plt.figure(figsize=(8, 4))
plt.plot(hist_f.history['accuracy'],     label='Treino')
plt.plot(hist_f.history['val_accuracy'], label='Validação')
plt.title('Acurácia — Fashion-MNIST')
plt.xlabel('Época')
plt.legend()
plt.show()

In [ ]:
# Predições em exemplos do teste
y_pred_f = np.argmax(model_f.predict(X_fte[:10], verbose=0), axis=1)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(X_fte[i], cmap='gray')
    color = 'green' if y_pred_f[i] == y_fte[i] else 'red'
    ax.set_title(f"Pred: {class_names[y_pred_f[i]]}\nReal: {class_names[y_fte[i]]}",
                 color=color, fontsize=8)
    ax.axis('off')
plt.suptitle('Predições (verde = correto, vermelho = errado)')
plt.tight_layout()
plt.show()